# 04.8 Multiple Assignment and Swapping

Python's assignment syntax goes well beyond `name = value`. Used well, unpacking
removes temporary variables, index arithmetic, and a whole class of off-by-one
errors.

## Theory

### Chained assignment

```python
a = b = c = 0
```

All three names are bound to **one object**, left to right. Harmless for
immutables; a trap for mutables — `a = b = []` gives two names for one list.

### Tuple unpacking

```python
x, y = 1, 2
```

The right-hand side is evaluated into a tuple first, then unpacked. The counts
must match, or you get a `ValueError`.

### The swap

```python
a, b = b, a
```

This works because the right side is **fully evaluated before any assignment
happens**. No temporary variable is required. You saw the bytecode in 04.1.

### Extended unpacking (PEP 3132)

```python
first, *rest = [1, 2, 3, 4]      # first=1, rest=[2, 3, 4]
```

The starred name collects everything not matched by the others. There can be at
most one star, and it always produces a **list**.

### Nested unpacking

The structure on the left can mirror any shape on the right:

```python
(name, (x, y)) = ("origin", (0, 0))
```

In [ ]:
# Chained assignment: one object, several names.
a = b = c = 0

print("Chained with an immutable:")
print("   a, b, c =", a, b, c)
print("   all the same object?", a is b is c)

# Rebinding one name does not affect the others.
a = 99
print("   after a = 99:", a, b, c)

# The same pattern with a MUTABLE object is a trap.
list_a = list_b = []
list_a.append("added via list_a")

print("")
print("Chained with a mutable:")
print("   list_a:", list_a)
print("   list_b:", list_b, "<- same object")
print("   same object?", list_a is list_b)

print("")
print("To get independent lists, assign separately:")
safe_a, safe_b = [], []
safe_a.append("only in safe_a")
print("   safe_a:", safe_a, " safe_b:", safe_b)

## Tuple unpacking

In [ ]:
# Basic unpacking - counts must match.
x_value, y_value = 10, 20
print("x, y =", x_value, y_value)

# Works with any iterable, not just tuples.
from_list = [1, 2, 3]
first, second, third = from_list
print("from a list:", first, second, third)

from_string = "abc"
letter_a, letter_b, letter_c = from_string
print("from a string:", letter_a, letter_b, letter_c)

# Mismatched counts raise a clear error.
print("")
try:
    only_two, = [1, 2, 3]
except ValueError as error:
    print("Too many values:", error)

try:
    one, two, three = [1, 2]
except ValueError as error:
    print("Too few values: ", error)

# A single-element unpack needs the trailing comma.
single, = [42]
print("")
print("single, = [42] gives:", single)

## The swap, and why it works

In [ ]:
# The classic swap.
left, right = "L", "R"
print("before:", left, right)

left, right = right, left
print("after: ", left, right)

# Three-way rotation works the same way.
one, two, three = 1, 2, 3
one, two, three = three, one, two
print("")
print("rotated:", one, two, three)

# It works because the RIGHT side is fully evaluated first.
values = [10, 20, 30]
index = 0

# Both the index and the value are computed before either is stored.
index, values[index] = 2, 99

print("")
print("index:", index)
print("values:", values, "<- 99 went to position 0, not 2")
print("")
print("The right side was evaluated first, so values[index] used the OLD")
print("index of 0. Assignments then happen left to right.")

## Extended unpacking with `*`

In [ ]:
numbers = [1, 2, 3, 4, 5]

# The star collects the remainder - always as a list.
first, *rest = numbers
print("first, *rest       ->", first, rest)

*most, last = numbers
print("*most, last        ->", most, last)

first, *middle, last = numbers
print("first, *middle, last ->", first, middle, last)

# The star can collect nothing.
only_one = [42]
single, *empty = only_one
print("")
print("single, *empty on [42] ->", single, empty, "<- empty list, not an error")

# Only one star is allowed.
print("")
try:
    compile("a, *b, *c = [1,2,3]", "<demo>", "exec")
except SyntaxError as error:
    print("Two stars:", error.msg)

## Nested unpacking

In [ ]:
# The left side can mirror any nested structure.
record = ("Asha", (1995, 7, 14), ["python", "sql"])

name, (year, month, day), skills = record

print("name:  ", name)
print("date:  ", year, month, day)
print("skills:", skills)

# Unpacking inside a loop - extremely common with dicts and pairs.
points = [(0, 0), (3, 4), (6, 8)]

print("")
print("Unpacking in a for loop:")
for x_coordinate, y_coordinate in points:
    # Each tuple is unpacked automatically.
    distance = (x_coordinate ** 2 + y_coordinate ** 2) ** 0.5
    print(f"   ({x_coordinate}, {y_coordinate}) is {distance:.1f} from origin")

# With enumerate, you unpack the counter and the item.
print("")
print("With enumerate:")
for position, (x_coordinate, y_coordinate) in enumerate(points, start=1):
    print(f"   point {position}: x={x_coordinate}, y={y_coordinate}")

## Where unpacking makes real code better

In [ ]:
# 1. Returning several values from a function.
def split_name(full_name):
    """Split a full name into first and last parts."""
    parts = full_name.split()
    # Returning a tuple lets the caller unpack it.
    return parts[0], parts[-1]


first_name, last_name = split_name("Ajay Nikumbh")
print("1. Multiple return values:", first_name, "/", last_name)

# 2. Iterating a dictionary's items.
settings = {"host": "localhost", "port": 8080}

print("")
print("2. Dict items:")
for key, value in settings.items():
    print(f"   {key:<6} {value}")

# 3. Parsing structured lines without index arithmetic.
csv_line = "2026-09-13,widget,4,19.99"
date, product, quantity, price = csv_line.split(",")

print("")
print("3. Parsing a CSV line:")
print(f"   date={date} product={product} qty={quantity} price={price}")

# 4. Ignoring values you do not need.
full_record = ("Asha", 31, "Mumbai", "engineer")
name, _, city, _ = full_record

print("")
print("4. Ignoring fields with _:", name, "from", city)

# 5. Swapping dict keys and values.
original = {"a": 1, "b": 2}
inverted = {value: key for key, value in original.items()}

print("")
print("5. Inverting a dict:", inverted)

## Takeaways

1. `a = b = c = 0` binds **one object** to several names — a trap for mutables.
2. Tuple unpacking evaluates the right side **completely first**, which is why
   `a, b = b, a` needs no temporary.
3. Counts must match, or you get a `ValueError` naming the mismatch.
4. `first, *rest` collects the remainder as a **list**; at most one star.
5. The left side can mirror any **nested** structure.
6. Unpacking in `for` loops removes index arithmetic and off-by-one errors.
7. Use `_` for values you deliberately ignore.

## Try it yourself

1. Predict then run: `a = b = []; a.append(1); print(b)`.
2. Swap three variables in one statement. Does the order work as you expect?
3. Unpack `[1,2,3,4,5]` four different ways using `*`.
4. Unpack `("x", (1, (2, 3)))` in a single statement.
5. Rewrite a loop of yours that uses `items[0]`, `items[1]` to use unpacking.